In [ ]:
!pip install sentence-transformers torch transformers

In [ ]:
import torch

# GPU가 연결되었는지 확인하는 코드
if torch.cuda.is_available():
    print(f"성공! 현재 사용 가능한 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU 연결 실패... CPU 모드입니다.")

In [ ]:
!pip install -U sentence-transformers

In [ ]:
import pandas as pd
import pickle

df = pd.read_csv("/content/drive/MyDrive/DX PROJ./층간소음데이터통합_영어제거완료.csv")

In [ ]:
df

In [ ]:
# 1. '통합본문'이 NaN인 행들만 필터링하여 새로운 변수에 담습니다.
nan_rows = df[df['본문'].isna()]

# 2. 결측치 행의 개수를 출력합니다.
print(f"현재 '본문'이 비어있는 행의 개수: {len(nan_rows)}개")

# 3. 결측치 행들을 화면에 출력합니다.
# 데이터가 많을 수 있으니 상위 10개 정도만 먼저 확인해 보세요.
nan_rows.head(10)

In [ ]:
# 1. '통합본문' 컬럼에 결측치가 있는 행을 제거합니다.
df = df.dropna(subset=['본문'])

In [ ]:
df

In [ ]:
df = df.drop(columns=['제목'])
display(df.head())

# 임베딩 및 결측치 제거 : 임베딩 해야할 열은 '본문'

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('jhgan/ko-sroberta-multitask')

In [ ]:
# 1. '통합본문' 컬럼에 결측치가 있는 행을 제거합니다.
df = df.dropna(subset=['본문'])

# 2. (혹시 모르니) 모든 데이터를 문자열 타입으로 확실히 변환합니다.
text_list = df['본문'].astype(str).tolist()

# 3. 다시 임베딩을 진행합니다.
print(f"총 {len(text_list)}개의 문장을 임베딩합니다...")
embeddings = model.encode(text_list, show_progress_bar=True)

# 4. 결과 확인
print("임베딩 완료! 형태:", embeddings.shape)

In [ ]:
df.to_pickle('KosentenceBERT_최종.pkl')

# sampling -> K-means

In [ ]:
from sklearn.cluster import MiniBatchKMeans
import numpy as np

# 1. 데이터를 1,000개의 대표점으로 요약합니다.
# n_clusters를 1000으로 설정합니다. (더 자세히 보고 싶다면 2000도 좋습니다.)
n_representative_points = 2000
kmeans_proxy = MiniBatchKMeans(n_clusters=n_representative_points, random_state=42, batch_size=2048)
kmeans_proxy.fit(embeddings)

# 2. 1,000개의 대표점 좌표(Centroids)를 추출합니다.
# 이제 이 'centroids'가 5만 개 데이터의 특징을 대변하는 요약본이 됩니다.
centroids = kmeans_proxy.cluster_centers_

print(f"압축 완료: {embeddings.shape[0]}개 데이터 -> {centroids.shape[0]}개 대표점")

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt

# 1. [누락되었던 핵심 코드] 대표점(centroids)을 이용해 연결 행렬 생성
# 군집 내 분산을 최소화하는 'ward' 연결 방식을 사용합니다.
linkage_matrix = sch.linkage(centroids, method='ward')

# 2. 덴드로그램 시각화 (작성하셨던 코드 그대로 사용)
plt.figure(figsize=(25, 12))

dendrogram = sch.dendrogram(
    linkage_matrix,
    truncate_mode='lastp',
    p=150,
    leaf_rotation=90.,
    leaf_font_size=8.,
    show_contracted=True
)

plt.title('Denser Dendrogram (150 Representative Clusters)')
plt.xlabel('Cluster Size')
plt.ylabel('Distance (Ward)')


plt.show()

# 최적의 클러스터 갯수 찾기 - 실루엣 분석

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# 1. 테스트할 군집 개수 범위 설정 (예: 2개부터 20개까지)
k_range = range(2, 11)
silhouette_avg = []

print("실루엣 점수 계산 시작 (대표점 1,000개 기준)...")

for k in k_range:
    # KMeans 모델 생성 및 학습 (대표점인 centroids 활용)
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(centroids)

    # 실루엣 점수 계산
    score = silhouette_score(centroids, cluster_labels)
    silhouette_avg.append(score)
    print(f"군집 수(k): {k}, 실루엣 점수: {score:.4f}")

# 2. 결과 시각화
plt.figure(figsize=(12, 6))
plt.plot(k_range, silhouette_avg, 'go-') # 녹색 점선 그래프
plt.title('Silhouette Method For Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range) # x축 간격을 1단위로 표시
plt.grid(True)
plt.show()

# clustering

In [ ]:
from sklearn.cluster import KMeans

# 1. k=4로 클러스터링 실행
kmeans4 = KMeans(n_clusters=5, init='k-means++', random_state=42)
df['cluster_k5'] = kmeans4.fit_predict(embeddings)

In [ ]:
print("--- [k=5] 군집별 분포 ---")
print(df['cluster_k5'].value_counts().sort_index())

In [ ]:
def check_cluster_samples(df, cluster_col, n_samples=5):
    clusters = sorted(df[cluster_col].unique())
    for c in clusters:
        print(f"\n{'='*20} {cluster_col} - Cluster {c} {'='*20}")
        # 해당 군집에서 샘플 추출
        samples = df[df[cluster_col] == c]['본문'].sample(n_samples, random_state=42)
        for i, text in enumerate(samples):
            print(f"[{i+1}] {text[:150]}...") # 내용을 150자까지 확인

# k=4 결과 먼저 확인하기
check_cluster_samples(df, 'cluster_k5')

In [ ]:
df['cluster_k5'].value_counts()

In [ ]:
# 1. 군집별로 텍스트를 합칩니다.
cluster_docs = df.groupby('cluster_k5')['본문'].apply(lambda x: ' '.join(x)).reset_index()

# 확인: cluster_docs는 4개의 행(0~3번 군집)을 가집니다.
print(cluster_docs.head())

In [ ]:
df.to_pickle('KosentenceBERT_최종_클러스터 5개로.pkl')

In [ ]:
import pickle

with open("KosentenceBERT_최종_클러스터 5개로.pkl", "rb") as file:
    df = pickle.load(file)

df.head()

In [ ]:
# 1. 군집별로 텍스트를 합칩니다.
cluster_docs = df.groupby('cluster_k5')['본문'].apply(lambda x: ' '.join(x)).reset_index()

# 확인: cluster_docs는 4개의 행(0~3번 군집)을 가집니다.
print(cluster_docs.head())

# tf-idf

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 2. TF-IDF 계산
# stop_words에 불필요한 단어(예: '진짜', '너무', '해요')를 추가하면 더 정확합니다.
tfidf = TfidfVectorizer(max_features=20) # 군집당 상위 20개 단어만 추출
tfidf_matrix = tfidf.fit_transform(cluster_docs['본문'])

# 단어 목록 가져오기
words = tfidf.get_feature_names_out()

In [ ]:
import pandas as pd

# 3. 각 군집별 상위 키워드 출력
importance = tfidf_matrix.toarray()

for i in range(4):
    # i번째 군집의 TF-IDF 점수가 높은 상위 10개 단어 인덱스 추출
    top_indices = importance[i].argsort()[-50:][::-1]
    top_keywords = [words[idx] for idx in top_indices]

    print(f"\n[Cluster {i}] 핵심 키워드:")
    print(", ".join(top_keywords))

In [ ]:
!pip install konlpy

# 단어 추출

In [ ]:
from konlpy.tag import Okt
import pandas as pd
from tqdm import tqdm

# 1. 형태소 분석기 초기화
okt = Okt()

# 2. 명사 추출 함수 정의
def extract_nouns(text):
    if not isinstance(text, str):
        return ""
    # 명사만 추출
    nouns = okt.nouns(text)
    # 한 글자 단어 제외 및 불용어 필터링 (필요시 추가)
    nouns = [n for n in nouns if len(n) > 1]
    return " ".join(nouns)

# 3. tqdm을 사용하여 진행률 확인하며 명사 추출 실행
# 데이터가 많으므로 apply 대신 리스트 컴프리헨션 + tqdm 조합을 권장합니다.
tqdm.pandas()
df['extracted_nouns'] = df['본문'].progress_apply(extract_nouns)

# 4. 결과 확인 (제대로 추출됐는지 상위 5개만 확인)
print(df[['본문', 'extracted_nouns']].head())

In [ ]:
# 1. 군집별로 추출된 명사들을 합칩니다.
cluster_docs = df.groupby('cluster_k5')['extracted_nouns'].apply(lambda x: ' '.join(x)).reset_index()

# 2. TF-IDF 계산
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50) # 상위 50개까지 확인
tfidf_matrix = tfidf.fit_transform(cluster_docs['extracted_nouns'])
words = tfidf.get_feature_names_out()
importance = tfidf_matrix.toarray()

# 3. 결과 출력
for i in range(5):
    top_indices = importance[i].argsort()[-20:][::-1]
    top_keywords = [words[idx] for idx in top_indices]
    print(f"\n[Cluster {i}] 핵심 키워드: {', '.join(top_keywords)}")

In [ ]:
top_n = 30
cluster_dict = {}

for i in range(len(cluster_docs)):
    # 해당 군집에서 TF-IDF 점수가 높은 상위 30개 인덱스 추출
    top_indices = importance[i].argsort()[-top_n:][::-1]
    cluster_dict[f'Cluster {i}'] = [words[idx] for idx in top_indices]

# 4. 딕셔너리를 데이터프레임으로 변환
keywords_df = pd.DataFrame(cluster_dict)

# 결과 확인
print("--- 군집별 상위 30개 핵심 키워드 ---")
display(keywords_df)

# ratio

In [ ]:
# 각 군집별 단어와 점수를 합친 데이터프레임 생성
summary_list = []

for i in range(len(cluster_docs)):
    top_indices = importance[i].argsort()[-30:][::-1]
    for rank, idx in enumerate(top_indices):
        summary_list.append({
            'Cluster': f'Cluster {i}',
            'Rank': rank + 1,
            'Keyword': words[idx],
            'Score': round(importance[i][idx], 4)
        })

# 보기 편하게 피벗 테이블 형태로 변환하거나 긴 형태로 유지
ranking_df = pd.DataFrame(summary_list)
ranking_df_pivot = ranking_df.pivot(index='Rank', columns='Cluster', values=['Keyword', 'Score'])

# 상위 15개 확인
display(ranking_df_pivot.head(15))

In [ ]:
import pandas as pd

# 1. 상위 30개 단어와 점수를 나란히 저장할 리스트 생성
top_n = 30
interleaved_data = {}

for i in range(len(cluster_docs)):
    # 해당 군집의 점수 높은 순으로 인덱스 추출
    top_indices = importance[i].argsort()[-top_n:][::-1]

    # 단어와 점수 컬럼을 각각 생성하여 저장
    interleaved_data[f'Cluster {i} Keyword'] = [words[idx] for idx in top_indices]
    interleaved_data[f'Cluster {i} Score'] = [round(importance[i][idx], 4) for idx in top_indices]

# 2. 데이터프레임 생성
final_keywords_score_df = pd.DataFrame(interleaved_data)

# 3. 인덱스를 Rank(순위)로 변경 (보기 편하게 1부터 시작)
final_keywords_score_df.index = [f'{i+1}위' for i in range(top_n)]

# 4. 결과 출력 및 파일 저장
print("--- [최종] 군집별 키워드 및 c-TF-IDF 점수 ---")
final_keywords_score_df.to_csv('cluster_keyword_scores_final.csv', encoding='utf-8-sig')
display(final_keywords_score_df)

# 공통단어 제거

In [ ]:
# 1. 제거하고 싶은 공통 단어 리스트 (이미지 결과를 보고 추가해 보세요)
custom_stopwords = [ '층간소음', '소음', '소리']

# 2. TfidfVectorizer 재설정
# max_df=0.75 : 4개 군집 중 3개 이상(75%)에서 공통으로 나오는 단어는 제외
tfidf = TfidfVectorizer(max_features=5000,
                        stop_words=custom_stopwords,
                        max_df=0.75)

tfidf_matrix = tfidf.fit_transform(cluster_docs['extracted_nouns'])
words = tfidf.get_feature_names_out()
importance = tfidf_matrix.toarray()

# 이후 동일하게 상위 30개 추출 및 데이터프레임 생성

In [ ]:
top_n = 10
interleaved_data = {}

for i in range(len(cluster_docs)):
    # 해당 군집의 점수 높은 순으로 인덱스 추출
    top_indices = importance[i].argsort()[-top_n:][::-1]

    # 단어와 점수 컬럼을 각각 생성하여 저장
    interleaved_data[f'Cluster {i} Keyword'] = [words[idx] for idx in top_indices]
    interleaved_data[f'Cluster {i} Score'] = [round(importance[i][idx], 4) for idx in top_indices]

# 4. 데이터프레임 생성 및 인덱스 설정
final_df = pd.DataFrame(interleaved_data)
final_df.index = [f'{i+1}위' for i in range(top_n)]

print("--- [최종] 정제된 군집별 키워드 및 영향력 점수 ---")
display(final_df) # 또는 print(final_df)

# 공통단어 제거 x

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. 필터(불용어, max_df)가 전혀 없는 순수 모델을 새로 생성합니다.
tfidf_raw = TfidfVectorizer(max_features=5000)

# 2. 군집별로 합쳐진 명사 데이터(cluster_docs)를 이 모델로 다시 학습시킵니다.
tfidf_matrix_raw = tfidf_raw.fit_transform(cluster_docs['extracted_nouns'])
words_raw = tfidf_raw.get_feature_names_out()
importance_raw = tfidf_matrix_raw.toarray()

# 3. 이미지(image_3c7904.png)와 똑같은 형식으로 데이터프레임 만들기
top_n = 100
original_view_data = {}

for i in range(len(cluster_docs)):
    # 점수가 높은 순서대로 인덱스 30개 추출
    top_indices = importance_raw[i].argsort()[-top_n:][::-1]

    # 단어 컬럼과 점수 컬럼을 나란히 생성
    original_view_data[f'Cluster {i} Keyword'] = [words_raw[idx] for idx in top_indices]
    original_view_data[f'Cluster {i} Score'] = [round(importance_raw[i][idx], 4) for idx in top_indices]

# 4. 데이터프레임 생성 및 인덱스 설정
original_df = pd.DataFrame(original_view_data)
original_df.index = [f'{i+1}위' for i in range(top_n)]

# 5. 결과 확인
print("--- [원본 버전] 공통 단어 포함 키워드 분석 ---")
display(original_df)

In [ ]:
original_df.to_csv('tfidf_keyword_results2.csv', index=True, encoding='utf-8-sig')

In [ ]:
df.to_pickle('KosentenceBERT_cluster5 최최종.pkl')

# topic 추출


In [ ]:
# 피클 형태를 불러오면, df['vector']가 텍스트가 아닌 원래의 배열 형태로 짠! 하고 나타남
import pandas as pd
df = pd.read_pickle('/content/drive/MyDrive/DX PROJ./임베딩/k=5/KosentenceBERT_cluster5 최최종.pkl')

In [ ]:
!pip install gensim
import gensim
from gensim import corpora, models
from gensim.corpora import Dictionary
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from tqdm import tqdm


In [ ]:
df['cluster_k5'].value_counts()

# 0번 클러스터 토픽으로 쪼개기

In [ ]:
import gensim
from gensim import corpora, models
from gensim.corpora import Dictionary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Cluster 0 데이터만 필터링하여 새로운 데이터프레임 생성
df_cluster0 = df[df['cluster_k5'] == 0].reset_index(drop=True)
print(f"🎯 Cluster 0 데이터 개수: {len(df_cluster0)}개")

# 2. Cluster 0의 임베딩 배열만 추출
X_c0 = np.stack(df_cluster0['embedding'].values)

# 3. 탐색할 하위 토픽 개수(K) 범위 설정
# 세부 토픽이므로 2~8개까지만 탐색합니다. (데이터 개수에 따라 조절 가능)
k_range_c0 = range(2, 9)

inertias_c0 = []
silhouette_scores_c0 = []

print("Cluster 0의 최적 하위 토픽 수(K)를 계산 중입니다...")

# 4. K값을 늘려가며 KMeans 학습 및 지표 계산
for k in k_range_c0:
    kmeans_c0 = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels_c0 = kmeans_c0.fit_predict(X_c0)

    # 엘보우 기법을 위한 Inertia(오차제곱합) 저장
    inertias_c0.append(kmeans_c0.inertia_)

    # 실루엣 점수 저장
    silhouette_scores_c0.append(silhouette_score(X_c0, labels_c0))

print("연산 완료! 결과를 그래프로 출력합니다.")

# 5. 시각화 (1행 2열)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# [왼쪽] 엘보우 기법 그래프
ax1.plot(k_range_c0, inertias_c0, marker='o', linestyle='-', color='blue')
ax1.set_title('Cluster 0: Elbow Method (Inertia)')
ax1.set_xlabel('Number of Sub-Topics (K)')
ax1.set_ylabel('Inertia')
ax1.set_xticks(k_range_c0)
ax1.grid(True)

# [오른쪽] 실루엣 점수 그래프
ax2.plot(k_range_c0, silhouette_scores_c0, marker='s', linestyle='-', color='red')
ax2.set_title('Cluster 0: Silhouette Score')
ax2.set_xlabel('Number of Sub-Topics (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(k_range_c0)
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 1. 변수를 데이터프레임의 열로 추가
df['embeddings'] = list(embeddings)

# 2. 확인
print(df.columns) # 'embeddings'가 있는지 확인